# 02 — Augmentation Strength Sweep

Trains 3 runs with label-aware augmentation at p ∈ {0.2, 0.4, 0.6}.

**What this notebook does:**
1. Trains one model per p value (all other hyperparameters identical)
2. Logs AP_easy, AP_mod, AP_hard, mAP@0.5 for each run
3. Plots all 4 metrics on the same figure (4 separate lines)
4. Identifies and records the optimal p

**Selection criterion:** highest AP_hard without degrading AP_easy by more than 2 points relative to p=0.2.

**Output:** Update `aug_probability` in configs M1–M8 with the selected p before training those variants.

**Prerequisite:** notebook `01_baseline` must have completed (M0 AP_hard is the reference).

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.logger import ExperimentLogger
from src.metrics import compute_kitti_ap, sample_to_annotation
from src.plotting import plot_augmentation_strength_sweep

ROOT = Path('..').resolve()
TABLES_DIR = ROOT / 'results' / 'tables'
FIGURES_DIR = ROOT / 'results' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuration ──────────────────────────────────────────────────────────────
RUN_MODE = 'local'   # change to 'kaggle' or 'colab' for full training
SMOKE_TEST = (RUN_MODE == 'local')

AUG_P_VALUES = [0.2, 0.4, 0.6]
P_CONFIG_MAP = {
    0.2: ROOT / 'configs' / 'aug_p02.yaml',
    0.4: ROOT / 'configs' / 'aug_p04.yaml',
    0.6: ROOT / 'configs' / 'aug_p06.yaml',
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ── Training loop across p values ──────────────────────────────────────────────
from ultralytics import YOLO

sweep_results = []  # list of dicts: {p, AP_easy, AP_mod, AP_hard, mAP50}

for p in AUG_P_VALUES:
    print(f'\n{'='*60}')
    print(f'Training aug sweep: p={p}')
    print(f'{'='*60}')

    cfg = load_config(P_CONFIG_MAP[p], overrides={
        'run_mode': RUN_MODE,
        'label_aware': True,
        'model_id': f'M_sweep_p{int(p*10):02d}',
        'epochs': 3 if SMOKE_TEST else 100,
        'data_limit': 100 if SMOKE_TEST else None,
    })
    cfg.ensure_dirs()
    set_all_seeds(cfg.seed)

    val_ds = KITTIDataset(cfg.kitti_root, split='val', imgsz=cfg.imgsz)
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch, shuffle=False,
        collate_fn=collate_fn, num_workers=0
    )

    model = YOLO('yolov8s.pt')
    logger = ExperimentLogger(
        run_id=f'sweep_p{p}_seed{cfg.seed}',
        log_dir=cfg.log_dir,
        config={'aug_p': p, **vars(cfg)},
    )

    with logger:
        model.train(
            data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),
            epochs=cfg.epochs,
            imgsz=cfg.imgsz,
            batch=cfg.batch,
            optimizer=cfg.optimizer,
            lr0=cfg.lr0,
            lrf=cfg.lrf,
            momentum=cfg.momentum,
            weight_decay=cfg.weight_decay,
            warmup_epochs=cfg.warmup_epochs,
            amp=cfg.amp,
            seed=cfg.seed,
            project=str(cfg.checkpoint_dir),
            name=cfg.model_id,
            exist_ok=True,
            device=device,
        )

        best_ckpt = cfg.checkpoint_dir / cfg.model_id / 'weights' / 'best.pt'
        best_model = YOLO(str(best_ckpt))
        best_model.model.eval()

        predictions, annotations = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(device)
                results = best_model(imgs, verbose=False)
                for i, r in enumerate(results):
                    boxes = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    scores = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    predictions.append({'image_id': batch['image_id'][i], 'boxes': boxes, 'scores': scores})
                    annotations.append(sample_to_annotation(batch, i))

        ap_easy = compute_kitti_ap(predictions, annotations, 'easy')
        ap_mod  = compute_kitti_ap(predictions, annotations, 'moderate')
        ap_hard = compute_kitti_ap(predictions, annotations, 'hard')

        # mAP@0.5 from Ultralytics val results
        val_metrics = best_model.val(data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),
                                     imgsz=cfg.imgsz, device=device, verbose=False)
        map50 = float(val_metrics.box.map50) * 100  # convert to percentage

        logger.log_metrics({'AP_easy': ap_easy, 'AP_mod': ap_mod, 'AP_hard': ap_hard, 'mAP50': map50})

    sweep_results.append({'p': p, 'AP_easy': ap_easy, 'AP_mod': ap_mod, 'AP_hard': ap_hard, 'mAP50': map50})
    print(f'p={p}  →  AP_easy={ap_easy:.2f}  AP_mod={ap_mod:.2f}  AP_hard={ap_hard:.2f}  mAP50={map50:.2f}')

sweep_df = pd.DataFrame(sweep_results)
print('\n', sweep_df.to_string(index=False))

In [ ]:
# ── Select optimal p ──────────────────────────────────────────────────────────
# Criterion: highest AP_hard without degrading AP_easy by more than 2 points
# relative to the p=0.2 run.

easy_at_p02 = sweep_df.loc[sweep_df['p'] == 0.2, 'AP_easy'].values[0]
AP_EASY_DROP_LIMIT = 2.0

eligible = sweep_df[sweep_df['AP_easy'] >= easy_at_p02 - AP_EASY_DROP_LIMIT]
optimal_row = eligible.loc[eligible['AP_hard'].idxmax()]
optimal_p = float(optimal_row['p'])

print(f'AP_easy at p=0.2: {easy_at_p02:.2f}')
print(f'Eligible p values (AP_easy drop ≤ {AP_EASY_DROP_LIMIT}): {eligible["p"].tolist()}')
print(f'\nSelected optimal p = {optimal_p}')
print(f'  AP_easy:  {optimal_row["AP_easy"]:.2f}')
print(f'  AP_mod:   {optimal_row["AP_mod"]:.2f}')
print(f'  AP_hard:  {optimal_row["AP_hard"]:.2f}')
print(f'  mAP50:    {optimal_row["mAP50"]:.2f}')
print(f'\nUpdate aug_probability={optimal_p} in configs M1–M8.')

In [ ]:
# ── Plot (Correction 6): AP_easy, AP_mod, AP_hard, mAP@0.5 as 4 separate lines ─
plot_augmentation_strength_sweep(
    sweep_data=sweep_df.to_dict('records'),
    optimal_p=optimal_p,
    save_dir=FIGURES_DIR,
    selection_note=(
        f'Selected p={optimal_p}: highest AP_hard without degrading '
        f'AP_easy by more than {AP_EASY_DROP_LIMIT} points'
    ),
)
print(f'Saved: {FIGURES_DIR / "aug_sweep.png"}')

In [ ]:
# ── Save sweep results ────────────────────────────────────────────────────────
sweep_df.to_csv(TABLES_DIR / 'aug_sweep_results.csv', index=False)
print(f'Saved: {TABLES_DIR / "aug_sweep_results.csv"}')

# Save optimal p so subsequent notebooks can load it
(ROOT / 'results').mkdir(exist_ok=True)
(ROOT / 'results' / 'optimal_aug_p.txt').write_text(str(optimal_p))
print(f'Optimal p={optimal_p} written to results/optimal_aug_p.txt')
print('\nACTION REQUIRED: Update aug_probability in configs/m1_blind_aug.yaml through m8 full_system.yaml')